In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch.nn.functional as F



# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train, dtype=torch.int64)
X_test  = torch.tensor(X_test, dtype=torch.int64)
y_train = torch.tensor(y_train, dtype=torch.int64)
y_test  = torch.tensor(y_test, dtype=torch.int64)



In [ ]:
# 2. Create TensorDataset objects
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)



In [ ]:
# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))


In [ ]:
# 4. Print shape of one batch
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")


In [ ]:
# 5. Display sample images
images, ages = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {ages[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()


In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # TODO: What are the dimensions from previous and next layer?
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)# <YOUR CODE HERE>

        # TODO: What are the dimensions from previous and next layer?
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)# <YOUR CODE HERE>

        # TODO: What are the dimensions of previous and output layer? (hint: hidden neurons? num of classes?)
        self.layer4 = nn.Linear(hidden_dim, output_dim)# <YOUR CODE HERE>

        # activation function for non-linearity
        self.relu = nn.ReLU()

    def forward(self, x):
        # Layer 1: TODO
        z1 = self.layer1(x)
        a1 = self.relu(z1)

        # Layer 2 TODO
        z2 = self.layer2(a1)
        a2 = self.relu(z2)

        # Layer 3 TODO
        z3 = self.layer3(a2)
        a3 = self.relu(z3)


        # Output layer TODO
        z4 = self.layer4(a3)


        return z4

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # Forward pass
        outputs = outputs = model(X_batch) # <YOUR CODE HERE>
        # TODO: compute loss
        loss = criterion(outputs, y_batch)# <YOUR CODE HERE>

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # Forward pass
            outputs =  model(X_batch)# <YOUR CODE HERE>
            # TODO: compute loss
            loss = criterion(outputs, y_batch)# <YOUR CODE HERE>
            running_loss += loss.item()


    avg_loss = running_loss / len(test_loader)
    # TODO: how do we calculate accuracy?
    accuracy = correct / total# <YOUR CODE HERE>

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NN4Layer(input_dim=3* 32 * 32, hidden_dim=32, output_dim=1).to(device)
print("Model Architecture:\n")
print(model)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

In [ ]:
# Task 5: Start training for 20 epochs:
# Run Training
num_epochs = 20
train_losses = []
val_losses = []
val_MSE = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, vali_MSE = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_MSE.append(vali_MSE)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here:
images, ages = next(iter(test_loader)) #take one lab
# Move images to device
images = images.to(device)
ages = ages.to(device)

with torch.no_grad():
    # Flatten images
    outputs = model(images.view(images.size(0), -1))
    predictions = torch.argmax(outputs, dim=1)

# Move tensors back to CPU for plotting
images = images.cpu()
ages = ages.cpu()
predictions = predictions.cpu()

# Plot first 6 predictions
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze())
    plt.title(f"True: {ages[i]} | Pred: {predictions[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()